In [3]:
import pandas as pd

# Load the CSV file
file_path = "jama_formatted_questions.csv"
df = pd.read_csv(file_path)

# Print the column statistics
print("Column Names:")
print(df.columns)


Column Names:
Index(['link', 'question', 'opa', 'opb', 'opc', 'opd', 'diagnosis',
       'answer_idx', 'answer', 'explanation', 'field', 'actual_question', 'id',
       'formatted_question', 'gpt_direct_prediction',
       'gpt_no_bullet_direct_prediction',
       'gpto3_mini_no_bullet_direct_prediction', 'gpto3_reasoning'],
      dtype='object')


In [5]:
# prompt: use OpenAI GPT4o to extract first columns' paper title. My prompt: Please extract paper title from this sentence.

!pip uninstall -y openai
!pip install --upgrade openai

import openai
print(openai.__version__)
import pandas as pd

# Assuming 'senior_author' DataFrame is loaded and contains a column named 'Paper Title'

# Set your OpenAI API key
openai.# Replace with your actual API key

Found existing installation: openai 1.72.0
Uninstalling openai-1.72.0:
  Successfully uninstalled openai-1.72.0
     |████████████████████████████████| 646 kB 6.7 MB/s eta 0:00:01
1.75.0


In [18]:
import pandas as pd
import re

def number_sentences_strict(text):
    lines = text.strip().split("\n")
    numbered_lines = []
    count = 1
    stop_numbering = False

    for i, line in enumerate(lines):
        stripped = line.strip().lstrip("-").strip()  # remove any leading dash and spaces

        # If empty or looks like a prompt line, stop numbering
        if not stripped or re.search(r"What\s+(Is|Would|Should)\b.*\?", stripped, re.IGNORECASE):
            stop_numbering = True
            numbered_lines.append(stripped)
            continue

        # If it's a multiple-choice option (A: B: etc.), don't number
        if re.match(r"^[A-J]:", stripped):
            numbered_lines.append(stripped)
            continue

        # Number sentences before we hit the question/options
        if not stop_numbering:
            numbered_lines.append(f"{count}. {stripped}")
            count += 1
        else:
            numbered_lines.append(stripped)

    return "\n".join(numbered_lines)

# Apply to your dataframe
df["numbered_question"] = df["actual_question"].apply(number_sentences_strict)

df.to_csv("jama_formatted_questions.csv", index=False)


## Direct Comparison

In [6]:
import openai

In [7]:
def generate_direct_prediction(question, opa, opb, opc, opd):
    full_prompt = f"""
    The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

    {question}

    A. {opa}
    B. {opb}
    C. {opc}
    D. {opd}

    Your task:
    - Select the best answer.
    - If the question refers to a figure or image, disregard it and focus solely on the text.
    - Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
    - Do not explain. Do not repeat the question.
    """
    try:
        response = openai.chat.completions.create(
            model="o3-mini",
            messages=[{"role": "user", "content": full_prompt}]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"OpenAI API call failed: {e}")
        return "Error"


In [9]:
# Select the first 1035 rows for testing
df_subset = df.head(1036).copy()

# Apply GPT-4o direct prediction to each row using separate question and answer fields
df_subset["gpt_no_bullet_direct_prediction"] = df_subset.apply(
    lambda row: generate_direct_prediction(
        row["question"], row["opa"], row["opb"], row["opc"], row["opd"]
    ),
    axis=1
)

# Display the question and the model's predicted answer
pd.set_option("display.max_colwidth", None)
display(df_subset[["question", "opa", "opb", "opc", "opd", "gpt_no_bullet_direct_prediction"]])

KeyboardInterrupt: 

In [21]:
# --- GPTo1 logic (commented out for now) ---
df_subset["gpt4o_no_bullet_letter"] = df_subset["gpt_no_bullet_direct_prediction"].str.strip().str[0]

df_subset["gpt4o_no_bullet_correct"] = df_subset.apply(
    lambda row: "Correct" if row["gpt4o_no_bullet_letter"] == row["answer_idx"] else "Incorrect",
    axis=1
)

gpt4o_no_bullet_correct_count = (df_subset["gpt4o_no_bullet_correct"] == "Correct").sum()
gpt4o_no_bullet_total_count = df_subset["gpt4o_no_bullet_letter"].notna().sum()
gpt4o_no_bullet_accuracy = gpt4o_no_bullet_correct_count / gpt4o_no_bullet_total_count

print(f"GPTo1 Correct Predictions: {gpt4o_no_bullet_correct_count}")
print(f"GPTo1 Total Predictions: {gpt4o_no_bullet_total_count}")
print(f"GPTo1 Accuracy: {gpt4o_no_bullet_accuracy:.2%}")

GPTo1 Correct Predictions: 765
GPTo1 Total Predictions: 1034
GPTo1 Accuracy: 73.98%


In [20]:
# Assign the new column to the original df for the first 1034 rows
# df.loc[df_subset.index, "gpto1_direct_prediction"] = df_subset["gpto1_direct_prediction"]
# df.loc[df_subset.index, "gpt_no_bullet_direct_prediction"] = df_subset["gpt_no_bullet_direct_prediction"]
df.loc[df_subset.index, "gpto3_mini_no_bullet_direct_prediction"] = df_subset["gpt_no_bullet_direct_prediction"]

# Save the updated DataFrame back to the CSV file
df.to_csv("jama_formatted_questions.csv", index=False)


## Bullet Points Prediction

In [ ]:
import openai

def generate_direct_prediction(question_text):
    """
    Uses GPT-4o to predict the correct answer from a multiple-choice question embedded in a single string.
    Returns only the predicted answer in the format: 'B: Femoral artery murmur'
    """
    prompt = f"""
    The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

    {question_text}

    Your task:
    - Select the best answer.
    - If the question refers to a figure or image, disregard it and focus solely on the text.
    - Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
    - Do not explain. Do not repeat the question.
    """

    try:
        # Initialize OpenAI client
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )
        content = response.choices[0].message.content.strip()
        return content
    except Exception as e:
        return "Error"


In [ ]:
# Example: first 1034 rows for testing
df_subset = df.head(1034).copy()

# Apply GPT-4o direct prediction to the 'actual_question' column
# df_subset["gpto1_direct_prediction"] = df_subset["actual_question"].apply(generate_direct_prediction)
df_subset["gpt_direct_prediction"] = df_subset["actual_question"].apply(generate_direct_prediction)

# Show results
pd.set_option("display.max_colwidth", None)
display(df_subset[["actual_question", "gpt_direct_prediction"]])
# display(df_subset[["actual_question", "gpto1_direct_prediction"]])

In [9]:
# Display the actual question and GPT model predictions
display(df_subset[["actual_question", "gpt_direct_prediction"]])

# Copy ground truth answer index
df_subset["answer_idx"] = df.loc[df_subset.index, "answer_idx"]

# Extract the first letter of the prediction as the predicted answer ID
df_subset["gpt_letter"] = df_subset["gpt_direct_prediction"].str.strip().str[0]

# Compare to ground truth to assign correctness
df_subset["gpt_correct"] = df_subset.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_idx"] else "Incorrect",
    axis=1
)

# Compute accuracy for GPT
gpt_correct_count = (df_subset["gpt_correct"] == "Correct").sum()
gpt_total_count = df_subset["gpt_correct"].notna().sum()
gpt_accuracy = gpt_correct_count / gpt_total_count

# Print results
print(f"GPT Correct Predictions: {gpt_correct_count}")
print(f"GPT Total Predictions: {gpt_total_count}")
print(f"GPT Accuracy: {gpt_accuracy:.2%}")


,actual_question,gpt_direct_prediction
0,"- A man in his 30s with AIDS presented with acute-onset painful scattered umbilicated papulopustules and ovoid ulcerated plaques with elevated, pink borders on the face, trunk, and extremities (Figure, A).\n- The patient also had a new-onset cough but was afebrile and denied other systemic symptoms.\n- Due to his significant immunocompromise, the clinical presentation was highly suspicious for infection.\n- For rapid bedside differentiation of multiple infectious etiologies, a Tzanck smear was performed by scraping the base of an ulcerated lesion and inner aspect of a pseudopustule and scraping its base with a #15 blade.\n- These contents were placed on a glass slide, fixed, and stained with Wright-Giemsa and subsequently Papanicolaou staining to further characterize the changes seen.A, Clinical image demonstrating papulopustules and ovoid ulcerated plaques with elevated, pink borders on the elbows.\n- B, Tzanck smear using Wright-Giemsa staining of specimen demonstrating ballooning of keratinocytes and peripheralization of nuclear material (original magnification ×20).\n\nWhat Is Your Diagnosis?\n\n- A: Herpes simplex virus\n- B: Histoplasmosis\n- C: Molluscum contagiosum\n- D: Mpox",A: Herpes simplex virus
1,"- An 80-year-old man with stage II bladder carcinoma (T2NXM0) and atrial fibrillation treated with apixaban presented to the emergency department with 1 week of fatigue and 2 days of dyspnea on exertion.\n- One week prior to presentation, he received a fourth cycle of carboplatin/gemcitabine for bladder carcinoma with 6 mg of pegylated granulocyte colony-stimulating factor (G-CSF).\n- The patient reported no anorexia, fever, melena, hematemesis, hematuria, cough, orthopnea, or peripheral edema.His vital signs were normal except for a heart rate of 103/min.\n- His white blood cell count was 22 × 103/μL (reference, 4-11 × 103/μL), increased from 4.8 × 103/μL 8 days prior.\n- His manual differential, which was previously normal, showed 18% bands (0%-10%), 2% metamyelocytes, 7% myelocytes, 7% promyelocytes, and 6% blasts.\n- His hemoglobin level was 5.2 g/dL (reference, 13-17 g/dL), decreased from 7.4 g/dL, and platelets were 25 × 103/μL (reference, 150-420 × 103/μL), decreased from 268 × 103/μL 8 days prior.\n- Ferritin was 1423 ng/mL (reference, 300-400 ng/mL).\n- Mean corpuscular volume, prothrombin time, international normalized ratio, partial thromboplastin time, fibrinogen, haptoglobin, vitamin B12, and methylmalonic acid values were normal, and results of a direct antiglobulin test were negative.\n- A computed tomography (CT) scan of his abdomen and pelvis was normal.\n- He received 2 units of packed red blood cells and was admitted to the hospital.\n- Flow cytometry identified a small population of CD34+/CD117+ cells (Figure).Left, Peripheral blood smear showing normocytic anemia with anisopoikilocytosis and leukocytosis with 6% to 8% blast forms.\n- Right, Flow cytometry of peripheral blood demonstrating a small population of white blood cells that stained positive for CD34 and CD117, which are markers of immature myeloblasts.Esophagogastroduodenoscopy revealed 2 nonbleeding angioectasias in the stomach that were treated with argon plasma coagulation.\n- Three days after admission, his white blood cell count was 27.7 × 103/μL with 4% peripheral blasts, hemoglobin was 7.3 g/dL, and platelet count had increased to 92 × 103/μL without a platelet transfusion.\n\nWhat Would You Do Next?\n\n- A: Perform a bone marrow biopsy\n- B: Prescribe all-trans retinoic acid\n- C: Repeat complete blood cell count with differential in 1 to 2 weeks\n- D: Start cytoreductive therapy with hydroxyurea",A: Perform a bone marrow biopsy
2,"- A 31-year-old man presented with left cervical and left inguinal masses.\n- He reported intermittent itching and night sweats for 2 years.\n- He denied fever, weight loss, shortness of breath, rashes, diarrhea, and neurological symptoms.\n- On a preemployment ev

GPT Correct Predictions: 705
GPT Total Predictions: 1034
GPT Accuracy: 68.18%


In [10]:
# Assign the new column to the original df for the first 1034 rows
# df.loc[df_subset.index, "gpto1_direct_prediction"] = df_subset["gpto1_direct_prediction"]
df.loc[df_subset.index, "gpt_direct_prediction"] = df_subset["gpt_direct_prediction"]

# Save the updated DataFrame back to the CSV file
df.to_csv("jama_formatted_questions.csv", index=False)


## o3-mini

In [15]:
import openai

def generate_direct_prediction(question_text):
    """
    Uses GPT-4o to predict the correct answer from a multiple-choice question embedded in a single string.
    Returns only the predicted answer in the format: 'B: Femoral artery murmur'
    """
    prompt = f"""
    The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

    {question_text}

    Your task:
    - Select the best answer.
    - If the question refers to a figure or image, disregard it and focus solely on the text.
    - Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
    - Do not explain. Do not repeat the question.
    """

    try:
        # Initialize OpenAI client
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="o3-mini", # gpt-4o
            messages=[{"role": "user", "content": prompt}]
        )
        content = response.choices[0].message.content.strip()
        return content
    except Exception as e:
        return "Error"


In [17]:
# Example: first 1034 rows for testing
df_subset = df.head(1035).copy()

# Apply GPT-4o direct prediction to the 'actual_question' column
df_subset["gpto1_direct_prediction"] = df_subset["actual_question"].apply(generate_direct_prediction)
# df_subset["gpt_direct_prediction"] = df_subset["actual_question"].apply(generate_direct_prediction)

# Show results
pd.set_option("display.max_colwidth", None)
# display(df_subset[["actual_question", "gpt_direct_prediction"]])
display(df_subset[["actual_question", "gpto1_direct_prediction"]])

,actual_question,gpto1_direct_prediction
0,"- A man in his 30s with AIDS presented with acute-onset painful scattered umbilicated papulopustules and ovoid ulcerated plaques with elevated, pink borders on the face, trunk, and extremities (Figure, A).\n- The patient also had a new-onset cough but was afebrile and denied other systemic symptoms.\n- Due to his significant immunocompromise, the clinical presentation was highly suspicious for infection.\n- For rapid bedside differentiation of multiple infectious etiologies, a Tzanck smear was performed by scraping the base of an ulcerated lesion and inner aspect of a pseudopustule and scraping its base with a #15 blade.\n- These contents were placed on a glass slide, fixed, and stained with Wright-Giemsa and subsequently Papanicolaou staining to further characterize the changes seen.A, Clinical image demonstrating papulopustules and ovoid ulcerated plaques with elevated, pink borders on the elbows.\n- B, Tzanck smear using Wright-Giemsa staining of specimen demonstrating ballooning of keratinocytes and peripheralization of nuclear material (original magnification ×20).\n\nWhat Is Your Diagnosis?\n\n- A: Herpes simplex virus\n- B: Histoplasmosis\n- C: Molluscum contagiosum\n- D: Mpox",A: Herpes simplex virus
1,"- An 80-year-old man with stage II bladder carcinoma (T2NXM0) and atrial fibrillation treated with apixaban presented to the emergency department with 1 week of fatigue and 2 days of dyspnea on exertion.\n- One week prior to presentation, he received a fourth cycle of carboplatin/gemcitabine for bladder carcinoma with 6 mg of pegylated granulocyte colony-stimulating factor (G-CSF).\n- The patient reported no anorexia, fever, melena, hematemesis, hematuria, cough, orthopnea, or peripheral edema.His vital signs were normal except for a heart rate of 103/min.\n- His white blood cell count was 22 × 103/μL (reference, 4-11 × 103/μL), increased from 4.8 × 103/μL 8 days prior.\n- His manual differential, which was previously normal, showed 18% bands (0%-10%), 2% metamyelocytes, 7% myelocytes, 7% promyelocytes, and 6% blasts.\n- His hemoglobin level was 5.2 g/dL (reference, 13-17 g/dL), decreased from 7.4 g/dL, and platelets were 25 × 103/μL (reference, 150-420 × 103/μL), decreased from 268 × 103/μL 8 days prior.\n- Ferritin was 1423 ng/mL (reference, 300-400 ng/mL).\n- Mean corpuscular volume, prothrombin time, international normalized ratio, partial thromboplastin time, fibrinogen, haptoglobin, vitamin B12, and methylmalonic acid values were normal, and results of a direct antiglobulin test were negative.\n- A computed tomography (CT) scan of his abdomen and pelvis was normal.\n- He received 2 units of packed red blood cells and was admitted to the hospital.\n- Flow cytometry identified a small population of CD34+/CD117+ cells (Figure).Left, Peripheral blood smear showing normocytic anemia with anisopoikilocytosis and leukocytosis with 6% to 8% blast forms.\n- Right, Flow cytometry of peripheral blood demonstrating a small population of white blood cells that stained positive for CD34 and CD117, which are markers of immature myeloblasts.Esophagogastroduodenoscopy revealed 2 nonbleeding angioectasias in the stomach that were treated with argon plasma coagulation.\n- Three days after admission, his white blood cell count was 27.7 × 103/μL with 4% peripheral blasts, hemoglobin was 7.3 g/dL, and platelet count had increased to 92 × 103/μL without a platelet transfusion.\n\nWhat Would You Do Next?\n\n- A: Perform a bone marrow biopsy\n- B: Prescribe all-trans retinoic acid\n- C: Repeat complete blood cell count with differential in 1 to 2 weeks\n- D: Start cytoreductive therapy with hydroxyurea",C: Repeat complete blood cell count with differential in 1 to 2 weeks
2,"- A 31-year-old man presented with left cervical and left inguinal masses.\n- He reported intermittent itching and night sweats for 2 years.\n- He denied fever, weight loss, shortness of breath, rashes, diarrhea, and neurolo

In [18]:
# --- GPTo1 logic (commented out for now) ---
df_subset["gpto1_letter"] = df_subset["gpto1_direct_prediction"].str.strip().str[0]

df_subset["gpto1_correct"] = df_subset.apply(
    lambda row: "Correct" if row["gpto1_letter"] == row["answer_idx"] else "Incorrect",
    axis=1
)

gpto1_correct_count = (df_subset["gpto1_correct"] == "Correct").sum()
gpto1_total_count = df_subset["gpto1_correct"].notna().sum()
gpto1_accuracy = gpto1_correct_count / gpto1_total_count

print(f"GPTo1 Correct Predictions: {gpto1_correct_count}")
print(f"GPTo1 Total Predictions: {gpto1_total_count}")
print(f"GPTo1 Accuracy: {gpto1_accuracy:.2%}")


GPTo1 Correct Predictions: 751
GPTo1 Total Predictions: 1034
GPTo1 Accuracy: 72.63%


## Randomize the Clinical Information Order

In [4]:
import random

def randomize_case_details(question_block):
    """
    Randomizes the lines of the clinical case except:
    - The first sentence (typically patient presentation)
    - The final question and answer options
    """
    # Split into lines and strip whitespace
    lines = [line.strip() for line in question_block.strip().split("\n") if line.strip()]
    
    # Find the index where the actual question begins (first line with '?')
    question_line_idx = next((i for i, line in enumerate(lines) if "?" in line), None)
    if question_line_idx is None:
        return question_block  # fail-safe: return original if no question found

    # First line: keep intact
    first_line = lines[0]

    # Lines to shuffle: lines between the first and the question line
    middle_lines = lines[1:question_line_idx]

    # Shuffle in-place
    random.shuffle(middle_lines)

    # Final block: reassemble all parts
    randomized_block = "\n".join([first_line] + middle_lines + lines[question_line_idx:])

    return randomized_block


In [5]:
df_subset = df.head(1036).copy()

df_subset["actual_question_randomized"] = df_subset["actual_question"].apply(randomize_case_details)
display(df_subset[["actual_question", "actual_question_randomized"]])

,actual_question,actual_question_randomized
0,- A man in his 30s with AIDS presented with ac...,- A man in his 30s with AIDS presented with ac...
1,- An 80-year-old man with stage II bladder car...,- An 80-year-old man with stage II bladder car...
2,- A 31-year-old man presented with left cervic...,- A 31-year-old man presented with left cervic...
3,- A 53-year-old woman with a history of stage ...,- A 53-year-old woman with a history of stage ...
4,- A 33-year-old man with no prior ocular probl...,- A 33-year-old man with no prior ocular probl...
...,...,...
1029,- A 2-year-old African American boy with no si...,- A 2-year-old African American boy with no si...
1030,- An otherwise healthy woman in her 50s was ev...,- An otherwise healthy woman in her 50s was ev...
1031,- A man aged 48 years was followed up for bila...,- A man aged 48 years was followed up for bila...
1032,- A young adult presented with a progressively...,- A young adult presented with a progressively...


In [7]:
import openai

def generate_direct_prediction(question_text):
    """
    Uses GPT-4o to predict the correct answer from a multiple-choice question embedded in a single string.
    Returns only the predicted answer in the format: 'B: Femoral artery murmur'
    """
    prompt = f"""
    The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

    {question_text}

    Your task:
    - Select the best answer.
    - If the question refers to a figure or image, disregard it and focus solely on the text.
    - Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
    - Do not explain. Do not repeat the question.
    """

    try:
        # Initialize OpenAI client
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="gpt-4o", # o3-mini
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )
        content = response.choices[0].message.content.strip()
        return content
    except Exception as e:
        return "Error"


In [8]:
# Apply GPT-4o to the randomized questions
df_subset["gpt4o_direct_prediction_randomized"] = df_subset["actual_question_randomized"].apply(generate_direct_prediction)

# Make sure long text is fully visible
import pandas as pd
pd.set_option("display.max_colwidth", None)

# Show a few examples with randomized prompt and prediction
display(df_subset[["actual_question_randomized", "gpt4o_direct_prediction_randomized"]].head())


KeyboardInterrupt: 

In [28]:
display(df_subset[["actual_question_randomized", "gpt4o_direct_prediction_randomized"]].head())


,actual_question_randomized,gpt4o_direct_prediction_randomized
0,"- A man in his 30s with AIDS presented with acute-onset painful scattered umbilicated papulopustules and ovoid ulcerated plaques with elevated, pink borders on the face, trunk, and extremities (Figure, A).\n- The patient also had a new-onset cough but was afebrile and denied other systemic symptoms.\n- B, Tzanck smear using Wright-Giemsa staining of specimen demonstrating ballooning of keratinocytes and peripheralization of nuclear material (original magnification ×20).\n- For rapid bedside differentiation of multiple infectious etiologies, a Tzanck smear was performed by scraping the base of an ulcerated lesion and inner aspect of a pseudopustule and scraping its base with a #15 blade.\n- These contents were placed on a glass slide, fixed, and stained with Wright-Giemsa and subsequently Papanicolaou staining to further characterize the changes seen.A, Clinical image demonstrating papulopustules and ovoid ulcerated plaques with elevated, pink borders on the elbows.\n- Due to his significant immunocompromise, the clinical presentation was highly suspicious for infection.\nWhat Is Your Diagnosis?\n- A: Herpes simplex virus\n- B: Histoplasmosis\n- C: Molluscum contagiosum\n- D: Mpox",A: Herpes simplex virus
1,"- An 80-year-old man with stage II bladder carcinoma (T2NXM0) and atrial fibrillation treated with apixaban presented to the emergency department with 1 week of fatigue and 2 days of dyspnea on exertion.\n- He received 2 units of packed red blood cells and was admitted to the hospital.\n- A computed tomography (CT) scan of his abdomen and pelvis was normal.\n- His white blood cell count was 22 × 103/μL (reference, 4-11 × 103/μL), increased from 4.8 × 103/μL 8 days prior.\n- One week prior to presentation, he received a fourth cycle of carboplatin/gemcitabine for bladder carcinoma with 6 mg of pegylated granulocyte colony-stimulating factor (G-CSF).\n- The patient reported no anorexia, fever, melena, hematemesis, hematuria, cough, orthopnea, or peripheral edema.His vital signs were normal except for a heart rate of 103/min.\n- Right, Flow cytometry of peripheral blood demonstrating a small population of white blood cells that stained positive for CD34 and CD117, which are markers of immature myeloblasts.Esophagogastroduodenoscopy revealed 2 nonbleeding angioectasias in the stomach that were treated with argon plasma coagulation.\n- His hemoglobin level was 5.2 g/dL (reference, 13-17 g/dL), decreased from 7.4 g/dL, and platelets were 25 × 103/μL (reference, 150-420 × 103/μL), decreased from 268 × 103/μL 8 days prior.\n- Ferritin was 1423 ng/mL (reference, 300-400 ng/mL).\n- Mean corpuscular volume, prothrombin time, international normalized ratio, partial thromboplastin time, fibrinogen, haptoglobin, vitamin B12, and methylmalonic acid values were normal, and results of a direct antiglobulin test were negative.\n- Flow cytometry identified a small population of CD34+/CD117+ cells (Figure).Left, Peripheral blood smear showing normocytic anemia with anisopoikilocytosis and leukocytosis with 6% to 8% blast forms.\n- His manual differential, which was previously normal, showed 18% bands (0%-10%), 2% metamyelocytes, 7% myelocytes, 7% promyelocytes, and 6% blasts.\n- Three days after admission, his white blood cell count was 27.7 × 103/μL with 4% peripheral blasts, hemoglobin was 7.3 g/dL, and platelet count had increased to 92 × 103/μL without a platelet transfusion.\nWhat Would You Do Next?\n- A: Perform a bone marrow biopsy\n- B: Prescribe all-trans retinoic acid\n- C: Repeat complete blood cell count with differential in 1 to 2 weeks\n- D: Start cytoreductive therapy with hydroxyurea",A: Perform a bone marrow biopsy
2,"- A 31-year-old man presented with left cervical and left inguinal masses.\n- Complete blood cell count and peripheral blood smear showed marked leukocytosis, with a white blood cell count of 22 340/μL, an absolute neutrophil count of 5360/μL, and 

In [29]:
# Create a new column with just the predicted answer letter (A/B/C/D)
df_subset["gpt_letter_randomized"] = df_subset["gpt4o_direct_prediction_randomized"].str.strip().str[0]

# Compare predictions with actual answers
df_subset["gpt_correct_randomized"] = df_subset.apply(
    lambda row: "Correct" if row["gpt_letter_randomized"] == row["answer_idx"] else "Incorrect",
    axis=1
)

# Accuracy stats
correct_count = (df_subset["gpt_correct_randomized"] == "Correct").sum()
total_count = df_subset["gpt_correct_randomized"].notna().sum()
accuracy = correct_count / total_count

# Print stats
print(f"Correct predictions: {correct_count}")
print(f"Total predictions: {total_count}")
print(f"Accuracy: {accuracy:.2%}")


Correct predictions: 676
Total predictions: 1034
Accuracy: 65.38%


In [ ]:
# Assign the new column to the original df for the first 1034 rows
df.loc[df_subset.index, "gpt4o_direct_prediction_randomized"] = df_subset["gpt4o_direct_prediction_randomized"]
df.loc[df_subset.index, "gpt_direct_prediction"] = df_subset["gpt_direct_prediction"]

# Randomized information re-run 
df.loc[df_subset.index, "gpt_direct_prediction_randomized"] = df_subset["gpt_direct_prediction_randomized"]


# Save the updated DataFrame back to the CSV file
df.to_csv("jama_formatted_questions.csv", index=False)


## W/ Reasoning


In [54]:
def get_reasoning_and_answer(actual_question):
    prompt = f"""
You are a highly capable and careful clinical reasoning assistant. Given a clinical vignette with multiple-choice options, think step by step to identify the most likely correct answer.

{actual_question}

First, explain your reasoning process in detail as if you were walking a student through the clinical logic step by step.

Then, at the end, write your final answer on a new line in the format:
Final Answer: <number>

Only use one of the digits 0, 1, 2, or 3 to indicate your final choice.
"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )

        content = response.choices[0].message.content.strip()

        if "Final Answer:" in content:
            parts = content.rsplit("Final Answer:", 1)
            reasoning = parts[0].strip()
            # Protect against empty answer part
            answer_part = parts[1].strip()
            answer = answer_part.split()[0] if answer_part else ""
        else:
            reasoning = content
            answer = ""

        return reasoning, answer

    except Exception as e:
        return "Error during generation", ""


In [55]:
# Run it on the first question
sample_question = df.loc[0, "actual_question"]
reasoning_output, final_answer = get_reasoning_and_answer(sample_question)

print("Reasoning Process:\n", reasoning_output)
print("\nPredicted Answer:", final_answer)

Reasoning Process:
 To solve this clinical vignette, let's go through the information step by step:

1. **Patient Demographics and Symptoms**: The patient is a man in his 30s with AIDS, which indicates significant immunocompromise. He presents with acute-onset painful scattered umbilicated papulopustules and ovoid ulcerated plaques with elevated, pink borders on the face, trunk, and extremities. He also has a new-onset cough but is afebrile and denies other systemic symptoms.

2. **Clinical Suspicion**: Given the patient's immunocompromised state, the presentation is highly suspicious for an infectious etiology.

3. **Diagnostic Test**: A Tzanck smear was performed. This test is commonly used to identify certain viral infections, particularly those caused by herpesviruses.

4. **Tzanck Smear Findings**: The smear showed ballooning of keratinocytes and peripheralization of nuclear material. These findings are characteristic of herpesvirus infections, where you often see multinucleated g

In [21]:
import os
if os.path.exists("jama4o_partial_results.csv"):
    os.remove("jama4o_partial_results.csv")


In [56]:
import pandas as pd
from tqdm.notebook import tqdm

# Initialize or resume from partial results
results_path = "jama4o_partial_results.csv"

try:
    results_df = pd.read_csv(results_path, index_col=0)
    done_indices = set(results_df.index)
    print(f"Resuming from {len(done_indices)} previously completed rows.")
except FileNotFoundError:
    results_df = pd.DataFrame(columns=["4o_reasoning_process", "4o_reasoning_answer"])
    done_indices = set()
    print("Starting fresh.")

# Loop with real-time progress and periodic saving
new_rows = []

for idx in tqdm(df.index):
    if idx in done_indices:
        continue

    question = df.at[idx, "actual_question"]
    
    try:
        reasoning, answer = get_reasoning_and_answer(question)
    except Exception as e:
        print(f"Error at index {idx}: {e}")
        reasoning, answer = None, None

    new_rows.append((idx, reasoning, answer))

    if len(new_rows) >= 10:  # Save every 10 rows
        temp_df = pd.DataFrame(new_rows, columns=["index", "4o_reasoning_process", "4o_reasoning_answer"])
        temp_df.set_index("index", inplace=True)
        results_df = pd.concat([results_df, temp_df])
        results_df.to_csv(results_path)
        new_rows = []

# Final save
if new_rows:
    temp_df = pd.DataFrame(new_rows, columns=["index", "4o_reasoning_process", "4o_reasoning_answer"])
    temp_df.set_index("index", inplace=True)
    results_df = pd.concat([results_df, temp_df])
    results_df.to_csv(results_path)

# Optionally merge into original DataFrame
df = df.join(results_df, how="left", rsuffix="_new")

# Save full version if needed
df.to_csv("jama4o_randomize_reasoning.csv", index=False)

Starting fresh.


  0%|          | 0/1034 [00:00<?, ?it/s]

In [58]:
# Define mapping from numeric prediction to letter label
number_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# Convert numeric string or integer predictions to letter form
results_df["gpt4o_reasoning_letter"] = results_df["4o_reasoning_answer"].astype(int).map(number_to_letter)

# Transfer the predicted letter to the main DataFrame based on index alignment
df["gpt4o_reasoning_letter"] = results_df["gpt4o_reasoning_letter"]

# Evaluate correctness by comparing predicted letter with ground-truth answer
df["gpt4o_reasoning_letter"] = df.apply(
    lambda row: "Correct" if row["gpt4o_reasoning_letter"] == row["answer_idx"] else "Incorrect",
    axis=1
)

# Compute accuracy statistics
correct_predictions = (df["gpt4o_reasoning_letter"] == "Correct").sum()
total_predictions = df["4o_reasoning_answer"].notna().sum()
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

# Report results
print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Prediction accuracy: {accuracy:.2%}")

# Display the first 20 comparisons for inspection
print("Ground-truth vs Predicted (first 20 rows):")
print(pd.DataFrame({
    "answer_idx": df["answer_idx"].head(20),
    "gpto3_reasoning_letter": df["gpt4o_reasoning_letter"].head(20)
}))


Correct predictions: 713
Total predictions: 1034
Prediction accuracy: 68.96%
Ground-truth vs Predicted (first 20 rows):
   answer_idx gpto3_reasoning_letter
0           D              Incorrect
1           C              Incorrect
2           D                Correct
3           C                Correct
4           B              Incorrect
5           C                Correct
6           C                Correct
7           B                Correct
8           B              Incorrect
9           C                Correct
10          B                Correct
11          D                Correct
12          D              Incorrect
13          B                Correct
14          D                Correct
15          B                Correct
16          B                Correct
17          A              Incorrect
18          D                Correct
19          C              Incorrect


In [50]:
# Define mapping from numeric prediction to letter label
number_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# Convert numeric string or integer predictions to letter form
results_df["gpto3_reasoning_letter"] = results_df["o3_mini_reasoning_answer"].astype(int).map(number_to_letter)

# Transfer the predicted letter to the main DataFrame based on index alignment
df["gpto3_reasoning_letter"] = results_df["gpto3_reasoning_letter"]

# Evaluate correctness by comparing predicted letter with ground-truth answer
df["gpto3_reasoning_answer"] = df.apply(
    lambda row: "Correct" if row["gpto3_reasoning_letter"] == row["answer_idx"] else "Incorrect",
    axis=1
)

# Compute accuracy statistics
correct_predictions = (df["gpto3_reasoning_answer"] == "Correct").sum()
total_predictions = df["gpto3_reasoning_answer"].notna().sum()
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

# Report results
print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Prediction accuracy: {accuracy:.2%}")

# Display the first 20 comparisons for inspection
print("Ground-truth vs Predicted (first 20 rows):")
print(pd.DataFrame({
    "answer_idx": df["answer_idx"].head(20),
    "gpto3_reasoning_letter": df["gpto3_reasoning_letter"].head(20)
}))


Correct predictions: 765
Total predictions: 1034
Prediction accuracy: 73.98%
Ground-truth vs Predicted (first 20 rows):
   answer_idx gpto3_reasoning_letter
0           D                      A
1           C                      C
2           D                      D
3           C                      D
4           B                      A
5           C                      C
6           C                      C
7           B                      B
8           B                      C
9           C                      C
10          B                      B
11          D                      D
12          D                      C
13          B                      B
14          D                      D
15          B                      B
16          B                      B
17          A                      A
18          D                      D
19          C                      A


In [25]:
# Assign the new column to the original df for the first 1034 rows
df.loc[df_subset.index, "gpto3_reasoning_result"] = df["gpto3_reasoning_letter"]

# # Add the predicted answer column based on actual_question
gpt_data["o3_reasoning_bullet_process"] = df["o3_mini_reasoning_process"]

# Save the updated DataFrame back to the CSV file
df.to_csv("jama_formatted_questions.csv", index=False)


NameError: name 'gpt_data' is not defined

## fetch_jama_cases.py

In [18]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime
from tqdm import tqdm
import pdb 

# URL of the JAMA Network Clinical Challenges page
BASE_URL = "https://jamanetwork.com/collections/44038/clinical-challenge" # 

# Function to scrape the clinical cases

def scrape_clinical_cases():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    cases_year = []
    page_number = 1
    
    while True:
        # Construct the URL for the current page
        url = f"{BASE_URL}?page={page_number}"
        response = requests.get(url, headers=headers)
        
        if response.status_code != 200:
            print(f"Failed to retrieve data from page {page_number}: {response.status_code}")
            break

        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Locate each clinical case and extract the link
        case_elements = soup.find_all("li", class_="article")  # General class to capture all JAMA medical field articles
        
        if not case_elements:
            print("No more cases found.")
            break
        
        for case in case_elements:
            link_tag = case.find("a", class_="article--title") 
            link = link_tag['href'] if link_tag else None

            # Extract the publication date
            date_tag = case.find("div", class_="article--date meta-item no-wrap")
            date_text = date_tag.text.strip() if date_tag else None
            
            # Parse the date and check the year
            if date_text:
                publication_date = datetime.strptime(date_text, "%B %d, %Y")
                # pdb.set_trace()
                if publication_date.year < 2013:
                    print("Reached articles older than 2013. Exiting.")
                    return cases_year  # Exit if the year is below 2013
            
            if link:
                cases_year.append((link,publication_date.year))

        print(f"Case URLs of page {page_number} fetched...")
        page_number += 1  # Move to the next page

        # if page_number > 2:
        #     break # for debugging
    return cases_year
    
# Function to extract answer_idx and answer from the clinical case page
def extract_answers(case_url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(case_url, headers=headers)
    if response.status_code != 200:
        return None, None  # Return None if there's an error

    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Locate the section for the diagnosis
    diagnosis_section = soup.find("div", class_="h4 cb section-type-section")
    
    # Extract the diagnosis title
    diagnosis_title = diagnosis_section.find("p", class_="para").text.strip() if diagnosis_section else None
    
    # Now, find all the answers following the diagnosis section
    answers = []
    for answer in soup.find_all("p", class_="para"):
        answer_text = answer.text.strip()
        if answer_text.startswith("A.") or answer_text.startswith("B.") or answer_text.startswith("C.") or answer_text.startswith("D."):
            answers.append(answer_text)

    # Assuming the first answer is the answer you want
    answer_idx = answers[0][0] if answers else None  # Get the first character (A, B, C, or D)
    answer = answers[0] if answers else None  # Get the full answer text
    answer = answer[3:] # convert 'D. Sternoclavicular sinus' to 'Sternoclavicular sinus'
    # pdb.set_trace()
    return answer_idx, answer

# Main execution
if __name__ == "__main__":
    clinical_cases_links = scrape_clinical_cases()
    
    compiled_results = []
    
    for idx, (link, year) in tqdm(enumerate(clinical_cases_links), total=len(clinical_cases_links)):
        answer_idx, answer = extract_answers(link)
        compiled_results.append({
            'id': idx,
            'link': link,
            'publication_year': year,
            'answer_idx': answer_idx,
            'answer': answer
        })
    
    # Save the results to a JSON file
    file_name = 'jama_links_updated.json'
    with open(file_name, 'w') as json_file:
        json.dump(compiled_results, json_file, indent=4)

    print(f"Saved {len(compiled_results)} cases to {file_name}")

Failed to retrieve data from page 1: 403


0it [00:00, ?it/s]

Saved 0 cases to jama_links_updated.json
